# 【語音辨識 - Whisper】 準確與否需要有一把 📏尺來衡量


前面我們介紹了幾個關於Whisper的基本概念，這裡附上 [🚀傳送門](https://vocus.cc/article/644526c8fd89780001ffdd9f) ，歡迎好好閱讀一番，但我們除了學會如何用語音辨識的工具之外，「準確率」對我們來說也是一個非常重要的一環，但我們究竟應該要如何評估所謂的準確率呢？ 不知道沒關係，當您看完這個篇章就能夠學會如何計算文字的「字元錯誤率」、「字詞錯誤率」...，非常值得您細細品嘗與學習，就讓我們往下一步步的完成評估準確率的程序吧！

這次的評估工具我們會使用jiwer這一套來進行說明，它支援了多種的計算方式，包括： WER、CER、MER...等，那這些計算方式各有什麼不同呢？ 就讓我們繼續看下去吧！

## 安裝套件

In [ ]:
# 錯誤率計算工具
!pip install jiwer

# 移除掉與語音辨識套件相同名稱的套件
# !pip uninstall whisper

# 語音辨識ASR
!pip install -U openai-whisper

# Hugging Face資料集函式庫
!pip install datasets

# 斷詞器
!pip install jiaba

!pip install requests

In [1]:

hypothesis = ' '.join(jieba.cut(hypothesis, cut_all=False, HMM=True))

out = jiwer.process_words(reference, hypothesis)
print(jiwer.visualize_alignment(out))

NameError: name 'jieba' is not defined

In [ ]:
import jieba

# Simple test function
def debug_chinese_tokenization(text):
    # Remove punctuation
    punct = "，。！？；：""''（）【】《》、…"
    for p in punct:
        text = text.replace(p, '')

    text = text.replace('\n', '')
    # Print text after punctuation removal
    print("After punctuation removal:", repr(text))

    # Tokenize and print words
    words = list(jieba.cut(text))
    print("After jieba tokenization:", words)

    # Create space-separated string
    result = ' '.join(words)
    print("Final result:", repr(result))
    return result

# Test with a small sample
sample_text = "今天天气很好。\n我想去公园走走。"
debug_chinese_tokenization(sample_text)

After punctuation removal: '今天天气很好我想去公园走走'
After jieba tokenization: ['今天天气', '很', '好', '我', '想', '去', '公园', '走走']
Final result: '今天天气 很 好 我 想 去 公园 走走'


'今天天气 很 好 我 想 去 公园 走走'

In [ ]:
from typing import List,Dict

In [ ]:
import jiwer
from jiwer import transforms
import jieba  # Chinese word segmentation library


# First, create a custom transformation for Chinese text
class ChineseTransform:
    def __init__(self):
        # Chinese punctuation marks to remove
        self.chinese_punc = "，。！？；：""''（）【】《》、…"

    def __call__(self, texts):
        newtexts = []
        for text in texts:
            # Remove Chinese and English punctuation
            for punct in self.chinese_punc:
                text = text.replace(punct, '')
            # Segment Chinese text into words using jieba
            words = jieba.cut(text, cut_all=False, HMM=True)
            # Join with spaces to make it compatible with jiwer
            newtext = ' '.join(words)
            print('after jieba cut:', newtext)
            newtexts.append(text)
        return newtexts



# Create transformation pipeline
transformation = transforms.Compose([
    #ChineseTransform(),  # Custom Chinese transformation
    transforms.Strip(),
    transforms.RemoveMultipleSpaces(),
])

def remove_newlines(texts):
    return ''.join( [text.strip() for text in texts.split('\n')])

def to_multi_lines(texts):
    return [text for text in texts.split('\n') if text.strip() != '']

# Example usage with Chinese text files
def calculate_chinese_wer( ground_truth:str, hypothesis:str):
    #ground_truth = remove_newlines(ground_truth)
    #hypothesis = remove_newlines(hypothesis)

    ground_truth_mlines =  to_multi_lines(ground_truth)
    hypothesis_mlines = to_multi_lines(hypothesis)

    ground_truth_mlines = chinese_transform(ground_truth_mlines)
    hypothesis_mlines = chinese_transform(hypothesis_mlines)

    # Calculate WER
    print('before wer')
    wer = jiwer.wer(
        ground_truth_mlines,
        hypothesis_mlines,
        truth_transform=transformation,
        hypothesis_transform=transformation
    )
    print('after wer')
    # Get detailed metrics
    measures = jiwer.compute_measures(
        ground_truth,
        hypothesis,
        truth_transform=transformation,
        hypothesis_transform=transformation
    )
    return wer, measures




In [ ]:
def read_files():
    refpath = 'sb_reference.txt'
    hypopath = 'sb_hypothesis.txt'
    nfstrs=[]
    for fpath in [refpath, hypopath]:
        with open(fpath, 'r', encoding='utf-8') as f:
            fstr = f.read()
        nfstrs.append(fstr)
    return nfstrs

def read_links():
    import requests
    hypolink='https://gist.github.com/timwu-ipevo/6a7fb0b6547e0d83b05a13e7d703e8ba/raw/f92a8efc257ba339af50bd68a1efad7449120a51/sb_hypothesis.txt'
    reflink='https://gist.github.com/timwu-ipevo/6a7fb0b6547e0d83b05a13e7d703e8ba/raw/f92a8efc257ba339af50bd68a1efad7449120a51/sb_reference.txt'

    refstr = requests.get(reflink).text
    hypostr = requests.get(hypolink).text
    return [refstr, hypostr]

In [ ]:
[refstr, hypostr] = read_links()

wer, measures = calculate_chinese_wer(
    refstr, hypostr
)

print(f"Word Error Rate: {wer:.3f}")
print(f"\nDetailed Metrics:")
print(f"Insertions: {measures['insertions']}")
print(f"Deletions: {measures['deletions']}")
print(f"Substitutions: {measures['substitutions']}")
print(f"Hits: {measures['hits']}")


after jieba cut: 大家 吉祥  
after jieba cut: 今天 要 跟 大家 談 人生 行路  
after jieba cut: 這是 出自 老舍 的 一個 文章  
after jieba cut: 老舍 本身 是 一位 文學家  
after jieba cut: 他 一生 可以 說 留下 很多  
after jieba cut: 好 的 小 說   好 的 文章  
after jieba cut: 讓 大家 都 非常 非常 的 受用 歡喜  
after jieba cut: 這 篇文章 裡面 有 講到  
after jieba cut: 才華是 刀刃   辛苦 是 磨刀石  
after jieba cut: 也 就是 我們 也 常講  
after jieba cut: 所謂 天才  
after jieba cut: 是 一分 的 天分   九十九 分 的 努力  
after jieba cut: 再鋒利 的 刀刃  
after jieba cut: 若日久 不磨 也 會 生銹  
after jieba cut: 也 表示 我們 在 日常生活 裡面  
after jieba cut: 不管 是 面 對 工作   面對 事業  
after jieba cut: 乃至 人 我 之間  
after jieba cut: 都 要 非常 的 努力  
after jieba cut: 都 要 很 專心 的 去 經營  
after jieba cut: 才 不會 從 中能夠 有所 差錯  
after jieba cut: 所以 在 這當 中 也 看出  
after jieba cut: 凡事 必須 要 勤勞    
after jieba cut: 凡事 也 必須 要 專心  
after jieba cut: 我們 講 一生 之計 在 於 勤  
after jieba cut: 這一輩子 不管 再 怎麼樣  
after jieba cut: 總是 要 勤勞  
after jieba cut: 我 也 經常講 一般 人  
after jieba cut: 一般 人以 為  
after jieba cut: 年輕 就是 本錢  
after jieba cut: 但事 實上  
after jieba cu

ValueError: After applying the transformation, each reference should be a non-empty list of strings, with each string being a single word.

In [ ]:
# rewrite ChineseTransform as a function
def chinese_transform(texts:List[str])->List[str]:
    chinese_punc = "，。！？；：""''（）【】《》、…"
    newtexts = []
    for text in texts:
        text = text.strip()
        # Remove Chinese and English punctuation
        for punct in chinese_punc:
            text = text.replace(punct, '')
        # Segment Chinese text into words using jieba
        words = jieba.cut(text, cut_all=False, HMM=False)
        # Join with spaces to make it compatible with jiwer
        newtext = ' '.join(words)
        newtexts.append(newtext)
    return newtexts



In [ ]:

def calculate_chinese_wer2( ground_truth:str, hypothesis:str, hslice):
    if True:
        ground_truth_mlines =  to_multi_lines(ground_truth)
        hypothesis_mlines = to_multi_lines(hypothesis)
        print('ground truth:')
        ground_truth_mlines = chinese_transform(ground_truth_mlines[hslice])
        print('hypothesis:')
        hypothesis_mlines = chinese_transform(hypothesis_mlines[hslice])

        ground_truth = ' '.join(ground_truth_mlines)
        hypothesis = ' '.join(hypothesis_mlines)


    out = jiwer.process_words( ground_truth, hypothesis ) #ground_truth_mlines ,hypothesis_mlines)
    print(jiwer.visualize_alignment(out))

[refstr, hypostr] = read_links()


In [ ]:
hypostr= hypostr.lstrip('敗')

In [ ]:
calculate_chinese_wer2( refstr, hypostr, slice(0,2))


ground truth:
hypothesis:
number of sentences: 1
substitutions=0 deletions=0 insertions=0 hits=9

mer=0.00%
wil=0.00%
wip=100.00%
wer=0.00%



In [ ]:
calculate_chinese_wer2( refstr, hypostr, slice(0,5))


ground truth:
hypothesis:
sentence 1
REF: 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 * 老舍 的 一 個 文章 老舍 本身 是 一位 文 學 家 他 一生 可以 說 留下 很多
HYP: 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 老  神 的 一 個 文章 老舍 本身 是 一位 文 學 家 * ** ** * ** **
                                    I  S                           D  D  D D  D  D

number of sentences: 1
substitutions=1 deletions=6 insertions=1 hits=23

mer=25.81%
wil=29.47%
wip=70.53%
wer=26.67%



In [ ]:
calculate_chinese_wer2( refstr, hypostr, slice(0,10))


ground truth:
hypothesis:
sentence 1
REF: 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 * 老舍 的 一 個 文章 老舍 本身 是 一位 文 學 家 他 一生 * 可以 說 留下 很多 好 的 小 說 好 的 文章 讓 大家 * 都 非常 非常 的 受用 歡 喜 * 這 篇文章 裡 面 有 講 到 才 華 是 刀刃 辛苦 是 ** 磨刀石 也 就是 我 們 也 常 講
HYP: 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 老  神 的 一 個 文章 老舍 本身 是 一位 文 學 家 他 一生 呢 可以 說 留下 很多 好 的 小 說 好 的 文章 讓 大家 呢 都 非常 非常 的 受用 歡 喜 那 這 篇文章 裡 面 有 講 到 才 華 是 刀刃 辛苦 是 磨刀   使 也 就是 * * * * *
                                    I  S                                I                                I                  I                                I   S      D D D D D

number of sentences: 1
substitutions=2 deletions=5 insertions=5 hits=60

mer=16.67%
wil=19.80%
wip=80.20%
wer=17.91%



In [ ]:
calculate_chinese_wer2( refstr, hypostr, slice(0,300))

ground truth:
hypothesis:
sentence 1
REF: 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 * 老舍 的 一 個 文章 老舍 本身 是 一位 文 學 家 他 一生 * 可以 說 留下 很多 好 的 小 說 好 的 文章 讓 大家 * 都 非常 非常 的 受用 歡 喜 * 這 篇文章 裡 面 有 講 到 才 華 是 刀刃 辛苦 是 ** 磨刀石 也 就是 我 們 也 常 講 所 謂 天才 * 是 * 一分 的 天分 九十九 分 的 努力 再 鋒 利 的 刀刃 若 日 久 不 磨 也 會 生 銹 也 表示 我 們 在 日常生活 裡 面 不管 是 面 對  工作 面 對 事 業 乃至 人 我 之 間 * 都 要 非常 的 努力 都 要 很 專 心 的 去 經 營 * 才 不 會 從 中 * 能 夠 有所 差 錯 所以 在 這 當 中 也 看出 ** 凡事 * 必 須 要 勤 勞 凡事 * 也 必 須 要 專 心 我 們 講 一生 之 計 * 在 於 勤 這 一 輩 子 不管 再 怎 麼 樣 總 是 要 勤 勞 我 也 經 常 講 一般 人 一般 人 以 為 年 輕 * 就是 本 錢 但 事 實 上 如果 說 一 個 年 輕 人 他 不 懂得 努力 不 懂得 勤 勞 我 想 * 那 也 非常 非常 可惜 所以 真正 的 本 錢 應 該 * 就是 我 們 一 顆 勤 勞 的 心 能 夠 面 對 種 種 的 事 面 對 種 種 的 人 也 就是 好好 的 去 經 營 好好 的 去 面 對 每 一 個 小 事情 你 都 很 用心 每 一 個 小 事情 * 你 都 * 不要 去 錯 過 我 想 這 樣 生活 * 確 實 能 夠 踏 實 * * 在 生活 裡 面 好 的 念 頭 也 非常 非常 重要 一 個 念 頭 我 們 或 許 會 覺 得 沒 什 麼 但是 好 的 念 頭 的 聚集 它 確 實 * 會 發 生 很大 的 力量 所以 我 們 常 講 細 水 長 流 穿破 * 石 熱 湯 停火 易 成 冰 這 也 表示 我 們 在 生活 裡 面 確 實 要 非常 非常 的 勤 勞 所 謂 滾 動 之 * 石 不易 生 苔 這 也 就是 我 們 * 生活 應 該 要 有 勤 勞 的 這 份 精神 謙 虛 使 人 的 心 縮 小 像 

我們先來看看詞的計算結果如下：

#### 詞錯誤率 Word Error Rate(WER)
WER是以「詞」為單位進行計算，它用來衡量句子中有多少詞彙需要進行修改才能和正確答案一樣。

```bash
公式: (S + D + I) / (H + S + D)
計算過程: (2 + 0 + 1) / (2 + 2 + 0)
3 / 4 ≈ 75%。
```

💡 既然是以`詞`為單位的話，那麼我們的答案與辨識結果請先進行斷詞(通常用空白隔開)， 標點符號也是考量的因素之一喔。

#### 平均錯誤率 Mean Error Rate(MER)
這項指標與WER主要差別在於分母的部分尚未將`Insertion`給考量進來計算，因為它衡量的不僅是詞彙層級，而是句子層級，因此會更加全面。

```bash
公式： (S + D + I) / (H + S + D + I)
計算過程： (2 + 0 + 1) / (2 + 2 + 0 + 1)

3 / 5 ≈ 60%
```

#### 詞保留率 Word Information Preservation(WIP)
這項指標主要在評估我們的辨識結果究竟有多少比例的字詞是一模一樣完全正確的。

```bash
num_rf_words = 正確答案字詞數 = 4
num_hp_words = 辨識結果字詞數 = 5
公式： (H / num_rf_words) * (H / num_hp_words)
計算過程: (2 / 4) * (2 / 5)
0.5 * 0.4 ≈ 20%
```
#### 詞漏失率 Word Information Lost(WIL)
既然有詞的保留率，那麼相反的就是漏失率，因此上述的結果得出之後，用1減去保留率就是漏失率，可以粗略的評估總共漏失了多少比率。
```bash
公式: 1 - wip
1 - 0.2 ≈ 80%
```

## 以「字元」為單位進行計算

### 字元錯誤率 Character Error Rate(CER)
CER是以「字元」為單位進行計算，底下的例子以「字元」為單位會發現有1個substitution，因此總共7個字元錯了1個等於：

```
1 / 7 = 14.28%
```

💡 既然是以`字元`為單位的話，那麼我們的答案與辨識結果請將空白給去除， 才不會也被計算進去喔， 甚至標點符號...等都是考量的因素之一。

In [ ]:
import jiwer

reference = "今天天氣很好嗎"
hypothesis = "今天天氣很好啊"

output = jiwer.process_characters(reference, hypothesis)
print(jiwer.visualize_alignment(output))

sentence 1
REF: 今天天氣很好嗎
HYP: 今天天氣很好啊
           S

number of sentences: 1
substitutions=1 deletions=0 insertions=0 hits=6

cer=14.29%



## 動動手使用Whisper語音辨識來計算一下正確率吧

首先我們先從Hugging Face找尋Common Voice的音檔：

https://huggingface.co/datasets/common_voice/viewer/zh-TW/train


### 接著我們使用Hugging Face的Datasets函式庫來進行操作

關於Datasets是什麼？ 歡迎參考：「[【Hugging Face】Ep.3 前往Datasets掏金趣](https://vocus.cc/article/64a2c62afd897800018a8185)」。

這邊會取測試集的第一筆做為我們的參考答案製作來源。

In [ ]:
from datasets import load_dataset

# 載入中文的資料集
ds = load_dataset("common_voice", name='zh-TW', split='test')

# 取第一筆做為本次的參考答案資料集
ref_data = ds[0]

ref_data

The repository for common_voice contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/common_voice.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


/root/.cache/huggingface/modules/datasets_modules/datasets/common_voice/3787625f0fa6c84d625ecc455d7fd8749f4c263ddcd48faab3ee972b44be9f84/common_voice.py:633: FutureWarning: 
            This version of the Common Voice dataset is deprecated.
            You can download the latest one with
            >>> load_dataset("mozilla-foundation/common_voice_11_0", "en")
            
  warnings.warn(


FileNotFoundError: https://voice-prod-bundler-ee1969a6ce8178826482b88e843c335139bd3fb4.s3.amazonaws.com/cv-corpus-6.1-2020-12-11/zh-TW.tar.gz

透過上述的操作我們可以得到幾個資訊：
- 語句: 並做出行動
- 音檔路徑: `/root/.cache/huggingface/datasets/downloads/extracted/0cd3800424a383996b39a64547fd7ea9852d200cc41113eebeb6f790cf74e9ca/cv-corpus-6.1-2020-12-11/zh-TW/clips/common_voice_zh-TW_17370757.mp3`

### 製作參考答案文字

首先我們對語句的部分進行斷詞，來製作參考答案的文字：

```python
並 做出 行動
```

In [ ]:
import jieba
_reference = ref_data['sentence']

reference = ' '.join(jieba.cut(_reference, cut_all=False, HMM=True))

reference # 兩國 總統 都 沒 有 直接 通過 電話


NameError: name 'ref_data' is not defined

### 接著來進行語音辨識

這邊我們使用`base`模型就好

In [ ]:
import whisper

model = whisper.load_model("base")

audio = ref_data['path']

result = model.transcribe(audio)

result

/usr/local/lib/python3.10/dist-packages/whisper/transcribe.py:114: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


{'text': '兩國總統都沒有直接通過電話。',
 'segments': [{'id': 0,
   'seek': 0,
   'start': 0.0,
   'end': 5.0,
   'text': '兩國總統都沒有直接通過電話。',
   'tokens': [50364,
    16313,
    8053,
    26575,
    33725,
    7182,
    6963,
    43297,
    19550,
    8816,
    20545,
    11103,
    1543,
    50614],
   'temperature': 0.0,
   'avg_logprob': -0.32310527165730796,
   'compression_ratio': 0.7924528301886793,
   'no_speech_prob': 0.031160296872258186}],
 'language': 'zh'}

### 最終我們透過jiwer計算一下錯誤率

正確率怎麼計算呢？ 1 - 錯誤率(WER、WIP、MER...)

P.S 看起來幾乎都沒有錯誤，猜測可能是common voice的語料已涵蓋在whisper的模型之中了...。

In [ ]:
import jiwer
hypothesis = result['text']

hypothesis = ' '.join(jieba.cut(hypothesis, cut_all=False, HMM=True))

out = jiwer.process_words(reference, hypothesis)
print(jiwer.visualize_alignment(out))

sentence 1
REF: 兩國 總統 都 沒 有 直接 通過 電話 *
HYP: 兩國 總統 都 沒 有 直接 通過 電話 。
                          I

number of sentences: 1
substitutions=0 deletions=0 insertions=1 hits=8

mer=11.11%
wil=11.11%
wip=88.89%
wer=12.50%

